In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4

cpu


In [14]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']


In [15]:
string_to_int = {ch:i for i, ch in enumerate(chars) }
int_to_string = {i:ch for i, ch in enumerate(chars) }

#Initialize Encoder and Decoder
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72, 60,
        63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73, 78,
        66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,  1,
        67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63, 31,
        73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77, 63,
         1, 73, 64,  1, 59, 72, 83, 73, 72, 63])


In [16]:
#Get training values
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    #Push to the GPU
    x,y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')

#Print(x.shape)
print(x)
print('targets:')
print(y)


tensor([122149, 105450, 140493, 148639])
inputs:
tensor([[78, 66, 67, 77,  1, 77, 74, 63],
        [77, 70, 73, 81, 70, 83,  1, 62],
        [78, 63, 72, 63, 62,  1, 73, 72],
        [77, 78, 13,  1, 60, 79, 78,  1]])
targets:
tensor([[66, 67, 77,  1, 77, 74, 63, 63],
        [70, 73, 81, 70, 83,  1, 62, 73],
        [63, 72, 63, 62,  1, 73, 72, 63],
        [78, 13,  1, 60, 79, 78,  1, 77]])


In [17]:

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, 'target is', target)

when input is tensor([49]) target is tensor(66)
when input is tensor([49, 66]) target is tensor(63)
when input is tensor([49, 66, 63]) target is tensor(1)
when input is tensor([49, 66, 63,  1]) target is tensor(45)
when input is tensor([49, 66, 63,  1, 45]) target is tensor(76)
when input is tensor([49, 66, 63,  1, 45, 76]) target is tensor(73)
when input is tensor([49, 66, 63,  1, 45, 76, 73]) target is tensor(68)
when input is tensor([49, 66, 63,  1, 45, 76, 73, 68]) target is tensor(63)


In [21]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embeddings_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets):
        logits = self.token_embeddings_table(index)
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        
        return logits
    